In [ ]:
!pip install econml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


In [ ]:
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print(f"Encoded confounder matrix shape: {X_encoded.shape}")

# --- Baseline 3: Linear DML (corrected) ---
ldml = LinearDML(model_y=HistGradientBoostingRegressor(random_state=42),
                  model_t=HistGradientBoostingRegressor(random_state=42),
                  discrete_treatment=False, cv=5, random_state=42)
ldml.fit(Y, T, X=X_encoded)
ldml_ate = ldml.ate(X_encoded)
ldml_ci = ldml.ate_interval(X_encoded)
print(f"Linear DML ATE: {ldml_ate:.4f}, 95% CI: [{ldml_ci[0]:.4f}, {ldml_ci[1]:.4f}]")

# --- Proposed: CausalForestDML (corrected) ---
cf = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=42),
                      model_t=HistGradientBoostingRegressor(random_state=42),
                      discrete_treatment=False, honest=True, min_samples_leaf=50,
                      n_estimators=1000, cv=5, random_state=42)
cf.fit(Y, T, X=X_encoded)
cf_ate = cf.ate(X_encoded)
cf_ci = cf.ate_interval(X_encoded)
cf_ite = cf.effect(X_encoded)
print(f"\nCausalForestDML ATE: {cf_ate:.4f}, 95% CI: [{cf_ci[0]:.4f}, {cf_ci[1]:.4f}]")
print(f"CATE distribution: mean={cf_ite.mean():.4f}, std={cf_ite.std():.4f}, "
      f"min={cf_ite.min():.4f}, max={cf_ite.max():.4f}")

Encoded confounder matrix shape: (17529, 35)
Linear DML ATE: 0.0029, 95% CI: [0.0026, 0.0033]

CausalForestDML ATE: 0.0027, 95% CI: [0.0011, 0.0042]
CATE distribution: mean=0.0027, std=0.0031, min=-0.0045, max=0.0073


In [ ]:
print(full['oucontent_clicks'].describe())
print(f"\n5th/95th percentile: {full['oucontent_clicks'].quantile([0.05, 0.95]).values}")

# Effect size in more interpretable terms
sd_t = full['oucontent_clicks'].std()
print(f"\nSD of oucontent_clicks: {sd_t:.1f}")
print(f"CausalForestDML effect per 1-SD increase in engagement: {cf_ate * sd_t:.3f} performance_gain points")
print(f"CausalForestDML effect per 100 additional clicks: {cf_ate * 100:.3f} performance_gain points")

count    17529.000000
mean       571.081066
std        781.729411
min          1.000000
25%         52.000000
50%        248.000000
75%        811.000000
max       9308.000000
Name: oucontent_clicks, dtype: float64

5th/95th percentile: [2.000e+00 2.152e+03]

SD of oucontent_clicks: 781.7
CausalForestDML effect per 1-SD increase in engagement: 2.075 performance_gain points
CausalForestDML effect per 100 additional clicks: 0.265 performance_gain points


In [ ]:
print(full['highest_education'].unique())

['HE Qualification', 'A Level or Equivalent', 'Lower Than A Level', 'Post Graduate Qualification', 'No Formal quals']
Categories (5, str): ['A Level or Equivalent', 'HE Qualification', 'Lower Than A Level', 'No Formal quals',
                      'Post Graduate Qualification']


In [ ]:
r, p_val = pearsonr(T, Y)
print(f"Pearson correlation: r={r:.4f}, p={p_val:.4g}")

X_ols = pd.get_dummies(X, drop_first=True)
ols_design = np.column_stack([T, X_ols.values])
ols = LinearRegression().fit(ols_design, Y)
print(f"OLS coefficient on treatment: {ols.coef_[0]:.4f}")

Pearson correlation: r=0.0794, p=6.143e-26
OLS coefficient on treatment: 0.0029


In [ ]:
edu_order = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent',
             'HE Qualification', 'Post Graduate Qualification']
full['highest_education'] = pd.Categorical(full['highest_education'], categories=edu_order, ordered=True)

print(full['highest_education'].value_counts().reindex(edu_order))

# CATE by education level, using the cf_ite already computed
full['cate'] = cf_ite
print("\nMean CATE by highest_education (ascending attainment):")
print(full.groupby('highest_education', observed=True)['cate'].agg(['mean', 'std', 'count']).reindex(edu_order))

highest_education
No Formal quals                 121
Lower Than A Level             5897
A Level or Equivalent          8228
HE Qualification               3059
Post Graduate Qualification     224
Name: count, dtype: int64

Mean CATE by highest_education (ascending attainment):
                                 mean       std  count
highest_education                                     
No Formal quals              0.001755  0.003259    121
Lower Than A Level           0.002810  0.003352   5897
A Level or Equivalent        0.002544  0.003143   8228
HE Qualification             0.002696  0.002382   3059
Post Graduate Qualification  0.002580  0.002536    224


In [ ]:
from scipy.stats import kruskal

groups = [full.loc[full['highest_education']==lvl, 'cate'].values for lvl in edu_order]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis H={stat:.2f}, p={p:.4g}")

from sklearn.model_selection import LeaveOneGroupOut

modules = full['code_module'].values
logo = LeaveOneGroupOut()

logo_results = []
for train_idx, test_idx in logo.split(X_encoded, Y, groups=modules):
    held_out_module = full['code_module'].iloc[test_idx].unique()[0]

    cf_fold = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=42),
                               model_t=HistGradientBoostingRegressor(random_state=42),
                               discrete_treatment=False, honest=True, min_samples_leaf=50,
                               n_estimators=500, cv=5, random_state=42)
    cf_fold.fit(Y[train_idx], T[train_idx], X=X_encoded.iloc[train_idx])
    ate_train = cf_fold.ate(X_encoded.iloc[train_idx])
    ate_heldout = cf_fold.ate(X_encoded.iloc[test_idx])

    logo_results.append({'held_out_module': held_out_module,
                          'ate_train_modules': ate_train, 'ate_heldout_module': ate_heldout})
    print(f"Held out {held_out_module}: train-modules ATE={ate_train:.4f}, held-out ATE={ate_heldout:.4f}")

logo_df = pd.DataFrame(logo_results)
print(f"\nHeld-out ATE range: [{logo_df['ate_heldout_module'].min():.4f}, {logo_df['ate_heldout_module'].max():.4f}]")
print(f"Mean absolute difference (train vs held-out): {(logo_df['ate_train_modules']-logo_df['ate_heldout_module']).abs().mean():.4f}")

NameError: name 'edu_order' is not defined

In [ ]:
!pip install econml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 13.4 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from econml.dml import CausalForestDML
from scipy.stats import kruskal

full = pd.read_csv('/content/drive/MyDrive/CausalMedia-GH/oulad_full_corpus.csv')

confounders = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
               'disability', 'code_presentation', 'code_module', 'num_of_prev_attempts', 'studied_credits']
categorical_cols = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
                     'disability', 'code_presentation', 'code_module']

full['imd_band'] = full['imd_band'].fillna('Missing')
X = full[confounders]
T = full['oucontent_clicks'].values
Y = full['performance_gain'].values
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

cf = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=42),
                      model_t=HistGradientBoostingRegressor(random_state=42),
                      discrete_treatment=False, honest=True, min_samples_leaf=50,
                      n_estimators=500, cv=5, random_state=42)
cf.fit(Y, T, X=X_encoded)
full['cate'] = cf.effect(X_encoded)

# Verify the column actually exists before doing anything else with it
assert 'cate' in full.columns, "cate column missing — CausalForestDML fit above did not complete"
print(f"cate column confirmed present. mean={full['cate'].mean():.4f}, n={len(full)}")

# Persist immediately, in this same cell, so this can't be lost again
full[['id_student', 'code_module', 'code_presentation', 'cate']].to_csv(
    '/content/drive/MyDrive/CausalMedia-GH/oulad_full_corpus_with_cate.csv', index=False)
print("Saved oulad_full_corpus_with_cate.csv")

# Now the Kruskal-Wallis test, in the same cell, using the same full/cate
edu_order = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent',
             'HE Qualification', 'Post Graduate Qualification']
full['highest_education'] = pd.Categorical(full['highest_education'], categories=edu_order, ordered=True)

groups = [full.loc[full['highest_education'] == lvl, 'cate'].values for lvl in edu_order]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis H={stat:.2f}, p={p:.4g}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cate column confirmed present. mean=0.0048, n=17529
Saved oulad_full_corpus_with_cate.csv
Kruskal-Wallis H=1250.82, p=1.528e-269


In [ ]:
k = len(edu_order)
n = len(full)
epsilon_sq = (1250.82 - k + 1) / (n - k)
print(f"Epsilon-squared (effect size): {epsilon_sq:.5f}")

Epsilon-squared (effect size): 0.07115


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut

modules = full['code_module'].values
logo = LeaveOneGroupOut()

logo_results = []
for train_idx, test_idx in logo.split(X_encoded, Y, groups=modules):
    held_out_module = full['code_module'].iloc[test_idx].unique()[0]

    cf_fold = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=42),
                               model_t=HistGradientBoostingRegressor(random_state=42),
                               discrete_treatment=False, honest=True, min_samples_leaf=50,
                               n_estimators=500, cv=5, random_state=42)
    cf_fold.fit(Y[train_idx], T[train_idx], X=X_encoded.iloc[train_idx])
    ate_train = cf_fold.ate(X_encoded.iloc[train_idx])
    ate_heldout = cf_fold.ate(X_encoded.iloc[test_idx])

    logo_results.append({'held_out_module': held_out_module,
                          'ate_train_modules': ate_train, 'ate_heldout_module': ate_heldout})
    print(f"Held out {held_out_module}: train ATE={ate_train:.4f}, held-out ATE={ate_heldout:.4f}")

logo_df = pd.DataFrame(logo_results)
logo_df.to_csv('/content/drive/MyDrive/CausalMedia-GH/logo_cv_results.csv', index=False)
print(f"\nSaved logo_cv_results.csv")
print(f"Held-out ATE range: [{logo_df['ate_heldout_module'].min():.4f}, {logo_df['ate_heldout_module'].max():.4f}]")
print(f"Mean abs difference (train vs held-out): {(logo_df['ate_train_modules']-logo_df['ate_heldout_module']).abs().mean():.4f}")

Held out AAA: train ATE=0.0028, held-out ATE=0.0036
Held out BBB: train ATE=0.0050, held-out ATE=0.0051
Held out CCC: train ATE=0.0028, held-out ATE=0.0040
Held out DDD: train ATE=0.0020, held-out ATE=0.0020
Held out EEE: train ATE=0.0025, held-out ATE=0.0039
Held out FFF: train ATE=-0.0029, held-out ATE=-0.0021

Saved logo_cv_results.csv
Held-out ATE range: [-0.0021, 0.0051]
Mean abs difference (train vs held-out): 0.0007


In [ ]:
print(full['code_module'].value_counts())

code_module
FFF    4930
BBB    4279
DDD    3608
CCC    2114
EEE    1986
AAA     612
Name: count, dtype: int64


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving studentVle.csv to studentVle.csv
Saving vle.csv to vle.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving studentAssessment.csv to studentAssessment.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving assessments.csv to assessments (1).csv
Saving studentRegistration.csv to studentRegistration.csv


In [ ]:
vle = pd.read_csv('vle.csv')
studentVle = pd.read_csv('studentVle.csv')

# Does GGG even have oucontent materials in the catalog?
ggg_vle = vle[vle['code_module'] == 'GGG']
print("GGG activity_type catalog:")
print(ggg_vle['activity_type'].value_counts())

# Does GGG have ANY studentVle click records at all?
ggg_clicks = studentVle[studentVle['code_module'] == 'GGG']
print(f"\nGGG total click records: {len(ggg_clicks)}")
print(f"GGG unique students with any click: {ggg_clicks['id_student'].nunique()}")

# Specifically, does GGG have oucontent clicks?
ggg_merged = ggg_clicks.merge(
    vle[['id_site', 'code_module', 'code_presentation', 'activity_type']],
    on=['id_site', 'code_module', 'code_presentation'], how='left'
)
print("\nGGG click volume by activity_type:")
print(ggg_merged.groupby('activity_type')['sum_click'].sum().sort_values(ascending=False))

# How many GGG students had valid outcomes (from the outcome-construction step)?
print(f"\nGGG students in full outcome table (final, before treatment merge): "
      f"{final[final['code_module']=='GGG']['id_student'].nunique() if 'final' in dir() else 'need to rebuild final first'}")

GGG activity_type catalog:
activity_type
resource     239
oucontent     75
quiz          26
subpage       15
forumng        6
homepage       3
glossary       3
Name: count, dtype: int64

GGG total click records: 387173
GGG unique students with any click: 2359

GGG click volume by activity_type:
activity_type
oucontent    477373
homepage     283454
forumng      239282
quiz         200337
resource      72615
subpage       52815
glossary       8008
Name: sum_click, dtype: int64

GGG students in full outcome table (final, before treatment merge): need to rebuild final first


In [ ]:
import pandas as pd

studentAssessment = pd.read_csv('/content/drive/MyDrive/CausalMedia-GH/studentAssessment.csv') \
    if False else pd.read_csv('studentAssessment.csv')
assessments = pd.read_csv('assessments.csv')
studentRegistration = pd.read_csv('studentRegistration.csv')

merged_assess = studentAssessment.merge(
    assessments[['id_assessment', 'code_module', 'code_presentation', 'assessment_type', 'date', 'weight']],
    on='id_assessment'
)

print(f"Stage 0 — GGG rows in merged_assess (all assessment attempts): "
      f"{len(merged_assess[merged_assess['code_module']=='GGG'])}")
print(f"Stage 0 — GGG unique students: {merged_assess[merged_assess['code_module']=='GGG']['id_student'].nunique()}")

# Stage 1: valid (non-banked, non-zero-weight) assessments
valid = merged_assess[(merged_assess['is_banked'] == 0) & (merged_assess['weight'] > 0)].copy()
ggg_valid = valid[valid['code_module']=='GGG']
print(f"\nStage 1 — GGG rows after is_banked/weight filter: {len(ggg_valid)}")
print(f"Stage 1 — GGG unique students: {ggg_valid['id_student'].nunique()}")

# Stage 2: >=2 valid assessments (trajectory feasibility)
valid_sorted = valid.sort_values('date')
first_last = valid_sorted.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    first_score=('score', 'first'), last_score=('score', 'last'),
    n_assessments=('score', 'size')
).reset_index()
ggg_before_filter = first_last[first_last['code_module']=='GGG']
print(f"\nStage 2 — GGG students before >=2 filter: {len(ggg_before_filter)}")
print(f"Stage 2 — GGG students' n_assessments distribution:")
print(ggg_before_filter['n_assessments'].value_counts().sort_index())

first_last = first_last[first_last['n_assessments'] >= 2].copy()
ggg_after_filter = first_last[first_last['code_module']=='GGG']
print(f"\nStage 2 — GGG students AFTER >=2 filter: {len(ggg_after_filter)}")

# Stage 3: withdrawal exclusion
first_last['performance_gain'] = first_last['last_score'] - first_last['first_score']
reg_small = studentRegistration[['id_student', 'code_module', 'code_presentation', 'date_unregistration']]
last_due = valid.groupby(['code_module', 'code_presentation'])['date'].max().rename('last_valid_due')
final = first_last.merge(reg_small, on=['id_student', 'code_module', 'code_presentation'], how='left')
final = final.merge(last_due, on=['code_module', 'code_presentation'], how='left')

ggg_before_withdrawal = final[final['code_module']=='GGG']
print(f"\nStage 3 — GGG students before withdrawal filter: {len(ggg_before_withdrawal)}")

final_after_withdrawal = final[~(final['date_unregistration'] < final['last_valid_due'])]
ggg_after_withdrawal = final_after_withdrawal[final_after_withdrawal['code_module']=='GGG']
print(f"Stage 3 — GGG students AFTER withdrawal filter: {len(ggg_after_withdrawal)}")

Stage 0 — GGG rows in merged_assess (all assessment attempts): 15219
Stage 0 — GGG unique students: 2107

Stage 1 — GGG rows after is_banked/weight filter: 0
Stage 1 — GGG unique students: 0

Stage 2 — GGG students before >=2 filter: 0
Stage 2 — GGG students' n_assessments distribution:
Series([], Name: count, dtype: int64)

Stage 2 — GGG students AFTER >=2 filter: 0

Stage 3 — GGG students before withdrawal filter: 0
Stage 3 — GGG students AFTER withdrawal filter: 0
